# Información del proyecto

## Propuesta inicial

En un contexto donde plataformas como Spotify o YouTube concentran gran parte del consumo musical, los artistas independientes enfrentan desafíos relacionados con la transparencia en los ingresos, la detección de usos no autorizados de su obra y la optimización de sus estrategias de lanzamiento. Este proyecto propone abordar estas problemáticas mediante el desarrollo de tres módulos principales basados en datos existentes. Los módulos son:

1. Análisis de mercado musical, orientado a identificar patrones de consumo, tendencias por género y oportunidades de monetización para artistas, permitiendo generar recomendaciones estratégicas basadas en datos (por ejemplo, mercados potenciales o características musicales asociadas al éxito).

2. Detección de fraude en streams, mediante técnicas de análisis de comportamiento y modelos de detección de anomalías, con el fin de identificar reproducciones artificiales o manipuladas que afectan la distribución justa de royalties.

3. Detector de copyright basado en inteligencia artificial, capaz de identificar la presencia de contenido musical en diferentes plataformas digitales mediante técnicas de procesamiento de audio, como espectrogramas y huellas digitales (audio fingerprinting), inspirado en tecnologías utilizadas por empresas como Shazam.

## Información de los módulos:



**1. Analisis del mercado musical apra el artista:**

* Datos posibles: Streams por país, Género musical, Playlist placement, TikTok trends, Crecimiento de artistas similares...

* Análisis (EDA): crecimiento por género, países con más crecimiento, correlación entre playlist y streams, horas/días con más reproducciones

* ML posible: Predicción de streams de una canción, Predicción de viralidad, 
* Modelos: Random Forest, XGBoost, LSTM para series temporales

* Output en tu plataforma

“Tu canción tiene 65% probabilidad de viralizarse en México”, “Este beat funciona mejor en playlist de chill trap”

* EXTRA DATOS: EDAD DE LA AUDIENCIA PARA AVERIGUAR TARGET SEGÚN PLATAFORMA. (MÉTRICAS AUDIENCIA TIK TOK)

**2. Fraude en streams:**

* Variables generales para objetivo 2:

| Variable                 | Tipo de dato | Para qué sirve                    | Ejemplo          |
| ------------------------ | ------------ | --------------------------------- | ---------------- |
| user_id                  | categorical  | identificar usuarios únicos       | U10233           |
| track_id                 | categorical  | identificar canción               | TRK8891          |
| session_id               | categorical  | identificar sesión                | S78123           |
| timestamp                | datetime     | detectar patrones temporales      | 2025-03-20 03:10 |
| listen_duration          | float        | detectar streams muy cortos       | 8.2 sec          |
| track_length             | float        | comparar duración real            | 180 sec          |
| completion_rate          | float        | ratio escucha completa            | 0.05             |
| device_type              | categorical  | bots usan dispositivos repetidos  | mobile / desktop |
| country                  | categorical  | detectar granjas de streams       | IN, RU           |
| playlist_source          | categorical  | detectar playlists sospechosas    | algorithmic      |
| repeat_count             | integer      | repeticiones consecutivas         | 25               |
| unique_tracks_in_session | integer      | bots suelen repetir pocas         | 1                |
| skip_rate                | float        | detectar skip masivo              | 0.90             |
| time_between_streams     | float        | bots tienen intervalos constantes | 5 sec            |


* Variables para futures engineering:

| Feature          | Fórmula                 |
| ---------------- | ----------------------- |
| repeat_ratio     | repeats / total streams |
| avg_listen_time  | mean(listen_duration)   |
| stream_frequency | streams / hour          |
| session_entropy  | diversidad de canciones |
| country_entropy  | diversidad geográfica   |


**3. Detector de copyright usando IA (Music Information Retrieval (MIR)):**

* Sistemas comerciales como Shazam usan huellas acústicas para identificar canciones.


* Modelo ML: con CNN usar como si fuera una imagen. 
* Al tratar info como imagen del sonido los ejes son: X = tiempo, Y = frecuencia y Color = energía.
* Variables generales para el punto 3: 

| Variable         | Tipo         | Para qué sirve                  | Ejemplo   |
| ---------------- | ------------ | ------------------------------- | --------- |
| track_id         | categorical  | identificar canción             | T123      |
| audio_file       | binary/audio | audio original                  | .wav      |
| sampling_rate    | int          | frecuencia audio                | 44100     |
| duration         | float        | duración track                  | 180s      |
| spectrogram      | matrix       | representación visual del audio | array     |
| mfcc_features    | vector       | características del timbre      | [12 dims] |
| chroma_features  | vector       | notas musicales                 | [12 dims] |
| tempo            | float        | BPM                             | 95        |
| pitch            | float        | tonalidad                       | C minor   |
| fingerprint_hash | hash         | huella digital del audio        | a9f12e    |


* Variables / Futures usadas en sonido:

| Feature            | Qué mide          |
| ------------------ | ----------------- |
| MFCC               | timbre del sonido |
| chroma             | notas musicales   |
| spectral centroid  | brillo del sonido |
| tempo              | BPM               |
| zero crossing rate | ruido             |


## Objetivos del proyecto



* Desarrollar un sistema de análisis de datos que permita comprender el comportamiento del mercado musical digital.

* Implementar modelos de machine learning para predecir el rendimiento de canciones y recomendar estrategias de monetización.

* Diseñar un sistema de detección de anomalías aplicado a datos de streaming para identificar posibles fraudes.

* Construir un prototipo de reconocimiento de audio para la identificación de contenido protegido por derechos de autor.

* Integrar estos componentes en una arquitectura escalable que pueda evolucionar hacia una plataforma real.

## Hipótesis de trabajo



* Es posible predecir el éxito relativo de una canción a partir de variables como sus características de audio, contexto de lanzamiento y métricas de engagement.

* Los patrones de fraude en streaming pueden ser identificados mediante comportamientos anómalos en los datos de escucha, sin necesidad de información sensible como direcciones IP.

* Las técnicas de procesamiento de audio permiten detectar coincidencias entre contenidos musicales con un alto grado de precisión, incluso en entornos ruidosos o transformados.

* El uso combinado de análisis de datos e inteligencia artificial puede ayudar a los artistas independientes a optimizar sus ingresos y tomar decisiones estratégicas más informadas.

# Data encontrada:


## **1. Analisis del mercado musical apra el artista:**

Se usan los datos de Last.fm API ya que  se basan en consumo real de usuarios.

### Imports

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

### API Request 

In [4]:
ret = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=chart.gettoptracks&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret.json()

{'tracks': {'track': [{'name': 'Stateside + Zara Larsson',
    'duration': '176',
    'playcount': '13739893',
    'listeners': '967585',
    'mbid': 'ffbf7862-2476-4164-ac32-f5904ccefe0f',
    'url': 'https://www.last.fm/music/PinkPantheress/_/Stateside+%252B+Zara+Larsson',
    'streamable': {'#text': '0', 'fulltrack': '0'},
    'artist': {'name': 'PinkPantheress',
     'mbid': '7441014f-f8f5-494f-81db-ff166fbc078d',
     'url': 'https://www.last.fm/music/PinkPantheress'},
    'image': [{'#text': 'https://lastfm.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'small'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'medium'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'large'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'extralarge'}]},
   {'name': 'Am

In [5]:
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")

Stateside + Zara Larsson . Duracion: 176 minutos
American Girls . Duracion: 213 minutos
Taste Back . Duracion: 221 minutos
Ready, Steady, Go! . Duracion: 160 minutos
Fame Is a Gun . Duracion: 183 minutos
Aperture . Duracion: 311 minutos
Babydoll . Duracion: 97 minutos
Coming Up Roses . Duracion: 248 minutos
Midnight Sun . Duracion: 189 minutos
Are you listening yet? . Duracion: 192 minutos
POP . Duracion: 216 minutos
Dracula . Duracion: 205 minutos
DtMF . Duracion: 237 minutos
The Waiting Game . Duracion: 169 minutos
Manchild . Duracion: 213 minutos
End of Beginning . Duracion: 159 minutos
Season 2 Weight Loss . Duracion: 229 minutos
Dance No More . Duracion: 194 minutos
Good Luck, Babe! . Duracion: 218 minutos
The Less I Know the Better . Duracion: 234 minutos
So Easy (To Fall In Love) . Duracion: 170 minutos
Carla's Song . Duracion: 253 minutos
WHERE IS MY HUSBAND! . Duracion: 196 minutos
back to friends . Duracion: 199 minutos
Creep . Duracion: 235 minutos
Paint By Numbers . Duracio

In [6]:
music = pd.DataFrame(ret.json()["tracks"]["track"])
music.head()

,name,duration,playcount,listeners,mbid,url,streamable,artist,image
0,Stateside + Zara Larsson,176,13739893,967585,ffbf7862-2476-4164-ac32-f5904ccefe0f,https://www.last.fm/music/PinkPantheress/_/Sta...,"{'#text': '0', 'fulltrack': '0'}","{'name': 'PinkPantheress', 'mbid': '7441014f-f...",[{'#text': 'https://lastfm.freetls.fastly.net/...
1,American Girls,213,1729748,350474,2c85fe70-3c0e-4b43-8d97-f5b5c4757f3a,https://www.last.fm/music/Harry+Styles/_/Ameri...,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35...",[{'#text': 'https://lastfm.freetls.fastly.net/...
2,Taste Back,221,1463030,280270,49562fdd-c325-415a-a2be-7d71f121b25d,https://www.last.fm/music/Harry+Styles/_/Taste...,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35...",[{'#text': 'https://lastfm.freetls.fastly.net/...
3,"Ready, Steady, Go!",160,1458007,289586,4f255861-925f-4629-ab41-22f4b35d8585,https://www.last.fm/music/Harry+Styles/_/Ready...,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35...",[{'#text': 'https://lastfm.freetls.fastly.net/...
4,Fame Is a Gun,183,18090157,969791,37907edc-06c1-4d08-b2ef-7e5221986f91,https://www.last.fm/music/Addison+Rae/_/Fame+I...,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Addison Rae', 'mbid': '610b71d9-fa78...",[{'#text': 'https://lastfm.freetls.fastly.net/...


In [7]:
# Cleaning the data frame:
music.drop(['mbid', 'image', 'url'],axis=1)

,name,duration,playcount,listeners,streamable,artist
0,Stateside + Zara Larsson,176,13739893,967585,"{'#text': '0', 'fulltrack': '0'}","{'name': 'PinkPantheress', 'mbid': '7441014f-f..."
1,American Girls,213,1729748,350474,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35..."
2,Taste Back,221,1463030,280270,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35..."
3,"Ready, Steady, Go!",160,1458007,289586,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35..."
4,Fame Is a Gun,183,18090157,969791,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Addison Rae', 'mbid': '610b71d9-fa78..."
5,Aperture,311,4674521,541196,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35..."
6,Babydoll,97,21490256,1447813,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Dominic Fike', 'mbid': 'e337c918-098..."
7,Coming Up Roses,248,1320572,260479,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35..."
8,Midnight Sun,189,10302086,635786,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Zara Larsson', 'mbid': '134e6410-695..."
9,Are you listening yet?,192,1281023,265197,"{'#text': '0', 'fulltrack': '0'}","{'name': 'Harry Styles', 'mbid': '7eb1ce54-a35..."


### Filtramos la información que queremos observar

In [8]:
# *Nombre y duración de las canciones
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")

Stateside + Zara Larsson . Duracion: 176 minutos
American Girls . Duracion: 213 minutos
Taste Back . Duracion: 221 minutos
Ready, Steady, Go! . Duracion: 160 minutos
Fame Is a Gun . Duracion: 183 minutos
Aperture . Duracion: 311 minutos
Babydoll . Duracion: 97 minutos
Coming Up Roses . Duracion: 248 minutos
Midnight Sun . Duracion: 189 minutos
Are you listening yet? . Duracion: 192 minutos
POP . Duracion: 216 minutos
Dracula . Duracion: 205 minutos
DtMF . Duracion: 237 minutos
The Waiting Game . Duracion: 169 minutos
Manchild . Duracion: 213 minutos
End of Beginning . Duracion: 159 minutos
Season 2 Weight Loss . Duracion: 229 minutos
Dance No More . Duracion: 194 minutos
Good Luck, Babe! . Duracion: 218 minutos
The Less I Know the Better . Duracion: 234 minutos
So Easy (To Fall In Love) . Duracion: 170 minutos
Carla's Song . Duracion: 253 minutos
WHERE IS MY HUSBAND! . Duracion: 196 minutos
back to friends . Duracion: 199 minutos
Creep . Duracion: 235 minutos
Paint By Numbers . Duracio

#### Comparación de grupos (canciones 'happy'-'sad'):

In [9]:
df_happy = music[music['name'].str.contains('happy',case=False,na=False)]
df_happy.count()

name          0
duration      0
playcount     0
listeners     0
mbid          0
url           0
streamable    0
artist        0
image         0
dtype: int64

In [10]:
df_sad = music[music['name'].str.contains('sad', case=False, na=False)]
df_sad.count()

name          0
duration      0
playcount     0
listeners     0
mbid          0
url           0
streamable    0
artist        0
image         0
dtype: int64

#### Duración promedio de top30 tracks:

In [11]:
# df_top30_duration = music.iloc[:30]
# df_top30_duration
# df_top30_duration.loc[:, 'duration'] = pd.to_numeric(df_top30_duration['duration'], errors='coerce')

# mean = df_top30_duration['duration'].mean()
# mean

### Procedencia de los artisitas top10:

#### Oyentes de los artistas top 10:

In [12]:
# df_top10 = music.head(10)
# df_top10
# df_top10['duration'] = pd.to_numeric(df_top10['duration'], errors='coerce')
# df_top10['listeners'] = pd.to_numeric(df_top10['listeners'], errors='coerce')

# df_top10['artist_name'] = df_top10['artist'].apply(lambda x: x['name'])

# plt.figure(figsize = (15, 5))

# plt.bar(df_top10['artist_name'], df_top10['listeners'], color = ['skyblue'])
# plt.xlabel('Artist name')
# plt.ylabel('Listeners count')
# plt.title("Listener of top 10 artist")
# plt.show()

In [13]:
# Info api request:
ret_artist = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=geo.gettopartists&country=spain&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret_artist.json()

{'topartists': {'artist': [{'name': 'Bad Bunny',
    'listeners': '13312',
    'mbid': '89aa5ecb-59ad-46f5-b3eb-2d424e941f19',
    'url': 'https://www.last.fm/music/Bad+Bunny',
    'streamable': '0',
    'image': [{'#text': 'https://lastfm.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'small'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'medium'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'large'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'extralarge'},
     {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png',
      'size': 'mega'}],
    '@attr': {'rank': '1'}},
   {'name': 'ROSALÍA',
    'listeners': '8847',
    'mbid': '25f3abd9-63b5-471a-bd25-feb9672dfa11',
    'url': 'https://www.last.fm/music/ROSA

In [14]:
artists_country = pd.DataFrame(ret_artist.json()["topartists"]["artist"])
artists_country.head()

,name,listeners,mbid,url,streamable,image,@attr
0,Bad Bunny,13312,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,https://www.last.fm/music/Bad+Bunny,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,{'rank': '1'}
1,ROSALÍA,8847,25f3abd9-63b5-471a-bd25-feb9672dfa11,https://www.last.fm/music/ROSAL%C3%8DA,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,{'rank': '2'}
2,Bad Gyal,8632,9cb2e99f-d0ba-4aa5-a371-0006b0d34090,https://www.last.fm/music/Bad+Gyal,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,{'rank': '3'}
3,Taylor Swift,8578,20244d07-534f-4eff-b4d4-930878889970,https://www.last.fm/music/Taylor+Swift,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,{'rank': '4'}
4,Charli xcx,8114,260b6184-8828-48eb-945c-bc4cb6fc34ca,https://www.last.fm/music/Charli+xcx,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,{'rank': '5'}


## Géneros musicales en tendencia global

In [ ]:

def get_trending_tags(api_key):
    url = f"https://ws.audioscrobbler.com/2.0/?method=tag.getTopTags&api_key={api_key}&format=json"
    response = requests.get(url)
    return response.json()['toptags']['tag']

api_key = '63e059c3c912a3f642daf2372484d183'
get_trending_tags(api_key)

[{'name': 'rock', 'count': 4069179, 'reach': 402894},
 {'name': 'electronic', 'count': 2498588, 'reach': 262197},
 {'name': 'seen live', 'count': 2194457, 'reach': 82557},
 {'name': 'alternative', 'count': 2130883, 'reach': 267196},
 {'name': 'pop', 'count': 2082364, 'reach': 233753},
 {'name': 'indie', 'count': 2065319, 'reach': 260434},
 {'name': 'female vocalists', 'count': 1634464, 'reach': 169157},
 {'name': 'metal', 'count': 1305603, 'reach': 159014},
 {'name': 'alternative rock', 'count': 1230035, 'reach': 170245},
 {'name': 'jazz', 'count': 1196398, 'reach': 149971},
 {'name': 'classic rock', 'count': 1149319, 'reach': 137113},
 {'name': 'ambient', 'count': 1125319, 'reach': 150168},
 {'name': 'experimental', 'count': 1111930, 'reach': 145396},
 {'name': 'folk', 'count': 958583, 'reach': 151351},
 {'name': 'indie rock', 'count': 928822, 'reach': 137423},
 {'name': 'punk', 'count': 919655, 'reach': 145492},
 {'name': 'Hip-Hop', 'count': 919195, 'reach': 131432},
 {'name': 'hard 

In [23]:
ret_generes = get_trending_tags(api_key)
top_generes = pd.DataFrame([ret_generes['name']['count']['reach']])
top_generes.head()

TypeError: list indices must be integers or slices, not str

## **2. Fraude en streams:**

Se usan los datos de Last.fm API también.

OBjetivo: Analizar el comportamiento de los usuarios para detectar anomalias. A través de la informaticón de timestamps y time_between_streams de podría averiguar si esa cuenta ussuario es un humano o un bot.

### Obtener el historial de un usuario para revisar anomalías

In [ ]:
# api_key = '63e059c3c912a3f642daf2372484d183'
# username = ''
# def check_user_activity(api_key, username):
#     url = f"https://ws.audioscrobbler.com/2.0/?method=user.getRecentTracks&user={username}&api_key={api_key}&format=json"
#     data = requests.get(url).json()
#     # Aquí podrías calcular la diferencia de tiempo entre tracks (feature engineering) [5]
#     return data['recenttracks']['track']


**NOTAS**

* Pipeline:

stream logs
↓
feature engineering
↓
EDA
↓
anomaly detection
↓
fraud score

* Modelo  ML debe buscar patrones como: 1000 streams, desde 1 usuario, en 1 hora, duración promedio 5 segundos

## **3. Detector de copyright usando IA (Music Information Retrieval (MIR)):**

Freesound proporciona descriptores de audio precomputados:

. Puedes obtener datos técnicos como: 

* MFCC (Mel-frequency cepstrum coefficients): Esenciales para la identificación de timbres

* Vectores de similitud (como LAION-CLAP): Diseñados para capturar propiedades acústicas y semánticas, ideales para sistemas de huella digital (audio fingerprinting)

. Análisis espectral: Incluye centroide espectral, complejidad y energía, que sirven para crear los espectrogramas que mencionas

Objetivo: Sound Analysis: 

1. En lugar de procesar el audio tú mismo desde cero, solicita directamente los descriptores mfcc, spectral_centroid y rhythm.bpm
2. Web Audio API: Úsala en el frontend de tu plataforma para que cuando el usuario suba una canción se genere el espectrograma visual

In [ ]:
# sound_id = 
# token = 
# def get_sound_features(sound_id, token):
#     # Solicitamos MFCC para el timbre y Spectral Centroid para el brillo [9, 10]
#     url = f"https://freesound.org/apiv2/sounds/{sound_id}/analysis/"
#     headers = {'Authorization': f'Token {token}'}
#     params = {'fields': 'mfcc,spectral_centroid,rhythm.bpm'}
    
#     response = requests.get(url, headers=headers, params=params)
#     return response.json()